# modeling_v13 — XGBoost 다중시드 lot-CV 검증 (R7 잠정→확정)

**목적** — M4에서 XGBoost 가 정직축(GKF, 단일 stable 파티션)에서 LGBM 을 **−0.6** 이긴 것이 **시드를 바꿔도 견고한지** 확인한다(PLAN §0.2 ①′(a) 다중시드 lot-CV · **R7 이중딥핑 통제**). 확정 셋 **Conservative-GA(99)** 고정, LGBM(기준) vs XGBoost.

> **전제 파일** (듀얼과 동일, 자동 탐색): `v13_fdc_pool_wf_oof.csv.gz` · `core10_meta_wf.csv` · `feature_diet_selected.json` · `select_result_Conservative_GA.json`
>
> **런타임(로컬 CPU)** : 4파티션(stable + seed 1/2/3) × 2모델 × 5fold = **40 fit** ≈ **~25분**. `Restart & Run All`.
>
> **판정 기준 (사전등록 · §6.2 / task)** : 3시드 개선폭 (LGBM−XGB) **평균 ≥ 0.5 ∧ 최악 시드 ≥ 0** → XGBoost 를 M5 §6.2 챔피언 후보로 **편입(승인 안건)**. 이 노트북은 **XGB vs LGBM 상대 우위의 견고성**만 확인(게이트 70.712 통과는 M5 튜닝 챔피언 대상 — 별개). **KFold 우위 불인정(R1).**

### 설계 (누수 안전 — 강건 점검)
- **다중시드 lot-CV**: 유니크 C20(1,217 lot)을 `KFold(5, shuffle=True, random_state=seed)` 로 fold 배정 → 샘플은 **자기 lot 의 fold 를 상속**. lot 이 fold 를 넘지 않음(= GKF 와 동일한 신규-Lot 누수 안전). seed ∈ {1,2,3}.
- **stable 파티션**: v2.2 `stable_group_kfold`(M4 메인 정직축) — 기준 대조로 함께 표기.
- 파티션은 numpy legacy RandomState(버전 안정)·stable 폴드맵 → **환경독립**. LGBM 은 로컬 정확 재현. **XGBoost 절대값만 로컬 버전 의존**(상대 격차 LGBM−XGB 는 동일환경 측정이라 견고).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, re, json, time
import numpy as np, pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# ── 동결 상수 ──
ID_COL, TARGET_COL = "C64", "C65"
CORE10 = ["is_high_regime", "high_regime_days", "days_since_last_pm", "C33",
          "dslp_x_hour", "hour", "hour_x_c33",
          "C60_mean_step4", "C59_mean_step4", "is_special_recipe"]
M8_PARAMS = dict(objective="regression", metric="rmse",
    learning_rate=0.029017547696366934, num_leaves=175, min_child_samples=5,
    feature_fraction=0.6324704159196377, bagging_fraction=0.864012693783303, bagging_freq=7,
    lambda_l1=5.04154328625296, lambda_l2=0.024814259264649002,
    min_split_gain=0.2573073648505903, verbose=-1, seed=42)
BEST_ROUNDS = 705
XGB_PARAMS = dict(n_estimators=BEST_ROUNDS, learning_rate=0.029, max_depth=8,
                  subsample=0.86, colsample_bytree=0.63, min_child_weight=5,
                  reg_lambda=0.02, reg_alpha=5.0, gamma=0.0,
                  tree_method="hist", random_state=42, n_jobs=-1, verbosity=0)
PROTECTED = ["C17", "C11", "C31", "C15", "C16"]
SEEDS = [1, 2, 3]
SIGMA_C65 = 261.7
# M4 stable 파티션 로컬 동결 (Cons-GA 99) — 대조 기준
REF_STABLE = {"LGBM": 71.366, "XGBoost": 70.773}
B0_REF = 71.212
ANCHOR = 70.712                                            # v2.3 게이트(챔피언 대상, 참고)

def _find(name):
    for d in [".", "data", "colab_GA", "..",
              os.path.join("..", "modeling_v13"),
              os.path.join("..", "modeling_v13", "data"),
              os.path.join("..", "modeling_v13", "colab_GA")]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음 — 노트북과 같은 폴더(또는 data/·colab_GA/)에 두세요.")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz")
META = _find("core10_meta_wf.csv")
DIET = _find("feature_diet_selected.json")
SEL_CONS = _find("select_result_Conservative_GA.json")
print("파일 확인:", *[os.path.basename(x) for x in [POOL, META, DIET, SEL_CONS]])

파일 확인: v13_fdc_pool_wf_oof.csv.gz core10_meta_wf.csv feature_diet_selected.json select_result_Conservative_GA.json


In [2]:
# ── 헬퍼 ──
def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def sensor_of(c):
    m = re.match(r"(C\d+)_", c)
    return m.group(1) if m else c

def floor_ok(feats):
    have = {s: sum(1 for c in feats if sensor_of(c) == s) for s in PROTECTED}
    return all(v >= 1 for v in have.values()), have

# v2.2 stable 폴드맵 (M4 메인 정직축)
def stable_group_kfold(groups, n_splits=5):
    groups = np.asarray(groups)
    _, inv, cnt = np.unique(groups, return_inverse=True, return_counts=True)
    order = np.argsort(cnt, kind="stable")[::-1]
    fold_sizes = np.zeros(n_splits, dtype=np.int64); g2f = np.zeros(len(cnt), dtype=np.int64)
    for gi in order:
        f = int(np.argmin(fold_sizes)); fold_sizes[f] += cnt[gi]; g2f[gi] = f
    return g2f[inv]

def stable_splits(groups, n_splits=5):
    fv = stable_group_kfold(groups, n_splits)
    return [(np.where(fv != k)[0], np.where(fv == k)[0]) for k in range(n_splits)]

# 다중시드 lot-CV: 유니크 lot 을 KFold(shuffle,seed) 로 fold 배정 → 샘플이 자기 lot fold 상속
def lot_seed_splits(groups, seed, n_splits=5):
    groups = np.asarray(groups)
    lots = np.unique(groups)                               # 정렬(안정)
    lot_fold = np.empty(len(lots), dtype=np.int64)
    for f, (_, te) in enumerate(KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(lots)):
        lot_fold[te] = f
    l2f = {l: int(fo) for l, fo in zip(lots, lot_fold)}
    fv = np.array([l2f[g] for g in groups])
    return [(np.where(fv != k)[0], np.where(fv == k)[0]) for k in range(n_splits)]

# OOF (동일 splits 리스트를 두 모델이 공유 → 공정 비교)
def oof_lgb(splits, df, feats, y):
    oof = np.zeros(len(df))
    for tr, va in splits:
        m = lgb.train(M8_PARAMS, lgb.Dataset(df.iloc[tr][feats], y[tr]), num_boost_round=BEST_ROUNDS)
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)

def oof_xgb(splits, df, feats, y):
    oof = np.zeros(len(df))
    for tr, va in splits:
        m = xgb.XGBRegressor(**XGB_PARAMS).fit(df.iloc[tr][feats], y[tr])
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)

def r2_honest(rmse):
    return round(1 - (rmse / SIGMA_C65) ** 2, 4)

In [3]:
# ── 로드 & 확정 셋 Conservative-GA(99) ──
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")
y = df[TARGET_COL].to_numpy(float); groups = df["C20"].to_numpy()
diet = json.loads(open(DIET, encoding="utf-8").read())
champions = list(diet["champions"]["Conservative"].values())
fixed = [f for f in dict.fromkeys(CORE10 + champions) if f in df.columns]
selp = json.loads(open(SEL_CONS, encoding="utf-8").read())["selected_features"]
feats = list(dict.fromkeys(fixed + selp))
ok, have = floor_ok(feats)
assert all(f in df.columns for f in feats) and ok and len(feats) == 99, f"셋 문제 n={len(feats)} floor={have}"
print(f"Conservative-GA 확정 셋 n={len(feats)}  floor={have}")
print(f"df {df.shape} | unique C20 {len(np.unique(groups))} lot | 시드 {SEEDS}")

Conservative-GA 확정 셋 n=99  floor={'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
df (11939, 667) | unique C20 1217 lot | 시드 [1, 2, 3]


In [4]:
# ── 4파티션(stable + seed 1/2/3) × 2모델 = 40 fit ──  (로컬 ~25분)
partitions = [("stable", stable_splits(groups))] + [(f"seed{s}", lot_seed_splits(groups, s)) for s in SEEDS]
rows = []
for name, splits in partitions:
    t = time.time()
    lg = oof_lgb(splits, df, feats, y)
    xg = oof_xgb(splits, df, feats, y)
    rows.append(dict(partition=name, LGBM=round(lg, 3), XGBoost=round(xg, 3),
                     delta_LGBM_minus_XGB=round(lg - xg, 3),
                     R2_LGBM=r2_honest(lg), R2_XGB=r2_honest(xg), sec=round(time.time() - t, 0)))
    print(f"{name:8s}  LGBM={lg:.3f}  XGB={xg:.3f}  Δ(LGBM−XGB)={lg-xg:+.3f}  ({time.time()-t:.0f}s)")

res = pd.DataFrame(rows)
res

stable    LGBM=71.366  XGB=70.773  Δ(LGBM−XGB)=+0.593  (133s)
seed1     LGBM=73.232  XGB=72.313  Δ(LGBM−XGB)=+0.919  (133s)
seed2     LGBM=69.870  XGB=69.444  Δ(LGBM−XGB)=+0.426  (133s)
seed3     LGBM=70.905  XGB=70.479  Δ(LGBM−XGB)=+0.426  (134s)


,partition,LGBM,XGBoost,delta_LGBM_minus_XGB,R2_LGBM,R2_XGB,sec
0,stable,71.366,70.773,0.593,0.9256,0.9269,133.0
1,seed1,73.232,72.313,0.919,0.9217,0.9236,133.0
2,seed2,69.870,69.444,0.426,0.9287,0.9296,133.0
3,seed3,70.905,70.479,0.426,0.9266,0.9275,134.0


In [5]:
# ── 판정 (사전등록 §6.2/task — 검산; 공식은 강건이 로컬 회신으로) ──
seed_rows = res[res["partition"].isin([f"seed{s}" for s in SEEDS])]
deltas = seed_rows["delta_LGBM_minus_XGB"].to_numpy(float)   # +면 XGB 우세
mean_d = float(deltas.mean()); min_d = float(deltas.min()); max_d = float(deltas.max())
all_positive = bool((deltas >= 0).all())                     # 최악 시드도 XGB≤LGBM
robust_edge  = all_positive and (min_d >= 0)
strong_half  = mean_d >= 0.5                                  # −0.5급 개선

print("=== XGBoost 다중시드 견고성 (Cons-GA 99) ===")
print(f"  stable 대조: Δ={float(res[res.partition=='stable'].delta_LGBM_minus_XGB.iloc[0]):+.3f}")
print(f"  시드 Δ(LGBM−XGB): {list(np.round(deltas,3))}  평균 {mean_d:+.3f}  최악 {min_d:+.3f}  최선 {max_d:+.3f}")
print(f"  [기준1] 방향 견고(전 시드 XGB≤LGBM · 최악≥0): {robust_edge}")
print(f"  [기준2] 평균 개선 ≥0.5: {strong_half}")
verdict = ("편입 권고(강)" if (robust_edge and strong_half)
           else "편입 권고(조건부·방향만)" if robust_edge
           else "편입 보류(견고성 부족)")
print(f"  → 검산 판정: {verdict}")
print("  ※ 게이트 70.712(v2.3) 통과는 M5 튜닝 챔피언 대상 — 본 노트북은 XGB vs LGBM 상대 견고성만.")

out = dict(set="Conservative-GA(99)", results=rows,
           seed_delta=dict(values=list(np.round(deltas, 3)), mean=round(mean_d, 3),
                           worst=round(min_d, 3), best=round(max_d, 3)),
           criterion=dict(robust_direction=robust_edge, mean_ge_0p5=strong_half, verdict=verdict),
           ref=dict(stable_LGBM=REF_STABLE["LGBM"], stable_XGB=REF_STABLE["XGBoost"],
                    B0=B0_REF, anchor_v23=ANCHOR),
           cv="lot-CV: KFold(5,shuffle,seed) on unique C20 + stable_group_kfold(대조)",
           note="XGB 절대값은 로컬 xgboost 버전 의존. 상대 Δ(LGBM−XGB)는 동일환경이라 견고. KFold 우위 불인정(R1).")
json.dump(out, open("xgb_multiseed_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("saved: xgb_multiseed_results.json")

=== XGBoost 다중시드 견고성 (Cons-GA 99) ===
  stable 대조: Δ=+0.593
  시드 Δ(LGBM−XGB): [np.float64(0.919), np.float64(0.426), np.float64(0.426)]  평균 +0.590  최악 +0.426  최선 +0.919
  [기준1] 방향 견고(전 시드 XGB≤LGBM · 최악≥0): True
  [기준2] 평균 개선 ≥0.5: True
  → 검산 판정: 편입 권고(강)
  ※ 게이트 70.712(v2.3) 통과는 M5 튜닝 챔피언 대상 — 본 노트북은 XGB vs LGBM 상대 견고성만.
saved: xgb_multiseed_results.json


---
### 판정 & 다음
- 이 노트북은 **XGB vs LGBM 상대 우위의 시드-견고성**(R7)만 본다. **게이트 70.712 통과 = M5 튜닝 챔피언 대상**(별개).
- **방향 견고(전 시드 XGB ≤ LGBM)** 이면 → XGBoost 를 **M5 §6.2 챔피언 후보로 편입**(LGBM·XGBoost 를 GKF 목적으로 동시 튜닝). §6.2 스코프 개정 = 사용자 승인.
- 견고성 부족(어떤 시드에서 XGB 가 LGBM 에 짐) 이면 → 단일 파티션 우위였을 가능성 → 편입 보류, 정직 기록.
- **KFold 우위는 불인정(R1)** — 본 노트북은 KFold 를 아예 재지 않는다(정직축 lot-CV 만).
- 공식 판정은 **로컬 실행 회신**으로 강건이 §6.2 기준 적용.